In [39]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import numpy as np, pandas as pd
from src.config import PROCESSED,ROOT
from src.indicators import atr
from src.strategies import señales_reversion
from src.features import crear_features, crear_dataset
import lightgbm as lgb

In [41]:
df = pd.read_parquet(PROCESSED)
df["atr"] = atr(df, 14)

In [42]:
señ = señales_reversion(df, horas=tuple(range(24)), k=1.5)          # regla primaria: todas las horas
feats = crear_features(df)
params = dict(sl_mult=2.0, tp_mult=1.5, max_velas=12, coste=0.25)
ds = crear_dataset(df, señ, feats, **params)

In [43]:
print(ds.shape, ds["y"].mean().round(3), ds["R"].mean().round(3))
ds.groupby(ds["datetime"].dt.year)[["y", "R"]].agg(["mean", "count"]).round(3)

(34086, 17) 0.542 -0.089


y            R      
           mean count   mean count
datetime                          
2020      0.560  4751 -0.063  4751
2021      0.533  4989 -0.124  4989
2022      0.539  5147 -0.117  5147
2023      0.547  5058 -0.114  5058
2024      0.536  5150 -0.104  5150
2025      0.536  5097 -0.063  5097
2026      0.546  3894 -0.023  3894

In [44]:



FEATURES = ["direccion", "z", "z_dia", "pend_dia", "atr_ratio", "ret_1", "ret_3", "ret_12",
            "rango_12", "coste_rel", "hora_sin", "hora_cos", "dia_sem"]
ds["año"] = ds["datetime"].dt.year

def nuevo_modelo():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15,
                              min_child_samples=200, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=0)

filas, importancias = [], []
for año in range(2022, 2027):
    inicio_test = pd.Timestamp(f"{año}-01-01")
    train = ds[ds["datetime"] < inicio_test - pd.Timedelta(days=2)]          # purga de 2 días
    test = ds[ds["año"] == año]

    modelo = nuevo_modelo().fit(train[FEATURES], train["y"])
    ultimo = train[train["año"] == año - 1]
    umbral = np.quantile(modelo.predict_proba(ultimo[FEATURES])[:, 1], 0.80)  # umbral sin mirar el test

    prob = modelo.predict_proba(test[FEATURES])[:, 1]
    elegidos = test[prob > umbral]

    # Control: filtro aleatorio del mismo tamaño
    rng = np.random.default_rng(0)
    azar = [test["R"].sample(len(elegidos), random_state=int(s)).mean()
            for s in rng.integers(0, 1e9, 200)]
    p = (np.array(azar) >= elegidos["R"].mean()).mean()

    filas.append({"año": año, "candidatas": len(test), "elegidas": len(elegidos),
                  "R_todas": test["R"].mean(), "R_ML": elegidos["R"].mean(),
                  "R_azar": np.mean(azar), "ventaja": elegidos["R"].mean() - np.mean(azar),
                  "p": p})
    importancias.append(pd.Series(modelo.feature_importances_, index=FEATURES, name=año))

res_ml = pd.DataFrame(filas).set_index("año")
res_ml.round(3)

,candidatas,elegidas,R_todas,R_ML,R_azar,ventaja,p
año,,,,,,,
2022,5147,1020,-0.117,-0.098,-0.118,0.020,0.21
2023,5058,858,-0.114,-0.087,-0.111,0.023,0.20
2024,5150,1178,-0.104,-0.091,-0.104,0.014,0.24
2025,5097,1348,-0.063,-0.057,-0.063,0.006,0.38
2026,3894,1091,-0.023,-0.006,-0.021,0.015,0.28


In [45]:
pd.concat(importancias, axis=1).mean(axis=1).sort_values(ascending=False)

rango_12     467.0
coste_rel    455.8
atr_ratio    450.0
z            442.0
ret_1        408.4
ret_12       404.8
ret_3        397.8
z_dia        342.2
pend_dia     296.6
hora_cos     214.2
hora_sin     202.2
dia_sem      117.4
direccion      1.6
dtype: float64

In [46]:
# "a favor del trade": positivo = el precio se mueve en la dirección del trade
for col in ["z", "z_dia", "pend_dia", "ret_1", "ret_3", "ret_12"]:
    ds[f"{col}_dir"] = ds[col] * ds["direccion"]

FEATURES_V2 = ["z_dir", "z_dia_dir", "pend_dia_dir", "ret_1_dir", "ret_3_dir", "ret_12_dir",
               "atr_ratio", "rango_12", "coste_rel", "hora_sin", "hora_cos", "dia_sem"]

In [47]:
def nuevo_regresor():
    return lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15,
                             min_child_samples=200, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=0)

filas, importancias, elegidos_todos = [], [], []
for año in range(2022, 2027):
    inicio_test = pd.Timestamp(f"{año}-01-01")
    train = ds[ds["datetime"] < inicio_test - pd.Timedelta(days=2)]
    test = ds[ds["año"] == año]

    modelo = nuevo_regresor().fit(train[FEATURES_V2], train["R"])           # predice la R esperada
    ultimo = train[train["año"] == año - 1]
    umbral = np.quantile(modelo.predict(ultimo[FEATURES_V2]), 0.95)         # top 5 %, sin mirar el test

    pred = modelo.predict(test[FEATURES_V2])
    elegidos = test[pred > umbral]
    elegidos_todos.append(elegidos)

    rng = np.random.default_rng(0)
    azar = [test["R"].sample(len(elegidos), random_state=int(s)).mean()
            for s in rng.integers(0, 1e9, 200)]
    filas.append({"año": año, "elegidas": len(elegidos),
                  "R_todas": test["R"].mean(), "R_ML": elegidos["R"].mean(),
                  "R_azar": np.mean(azar), "ventaja": elegidos["R"].mean() - np.mean(azar),
                  "p": (np.array(azar) >= elegidos["R"].mean()).mean()})
    importancias.append(pd.Series(modelo.booster_.feature_importance("gain"),
                                  index=FEATURES_V2, name=año))

res_v2 = pd.DataFrame(filas).set_index("año")
res_v2.round(3)

,elegidas,R_todas,R_ML,R_azar,ventaja,p
año,,,,,,
2022,307,-0.117,0.031,-0.116,0.147,0.000
2023,153,-0.114,-0.103,-0.113,0.009,0.440
2024,499,-0.104,-0.108,-0.102,-0.006,0.560
2025,586,-0.063,-0.055,-0.064,0.008,0.395
2026,469,-0.023,-0.028,-0.022,-0.005,0.575


In [48]:
el = pd.concat(elegidos_todos)
r = el["R"]
print(f"Conjunto OOS 2022–26: n={len(r)}  R_ML={r.mean():.3f}  "
      f"t={r.mean() / r.std() * np.sqrt(len(r)):.2f}  R_todas={ds[ds['año'] >= 2022]['R'].mean():.3f}")
pd.concat(importancias, axis=1).mean(axis=1).sort_values(ascending=False).round(0)

Conjunto OOS 2022–26: n=2014  R_ML=-0.052  t=-3.04  R_todas=-0.087


coste_rel       2118.0
z_dir           1491.0
rango_12        1491.0
ret_3_dir       1479.0
atr_ratio       1479.0
ret_12_dir      1388.0
ret_1_dir       1225.0
z_dia_dir       1083.0
pend_dia_dir     972.0
hora_cos         761.0
hora_sin         681.0
dia_sem          392.0
dtype: float64

In [49]:
from src.backtest import simular_trade_trailing
o, h, l, c = (df[col].values for col in ["open", "high", "low", "close"])
a = df["atr"].values

# 1) Un trade concreto: comprobarlo a mano con las velas
i = 200_000
print(simular_trade_trailing(o, h, l, c, i, 1, a[i-1], a[i-1], 24, 0.25))
df.iloc[i:i+24][["open", "high", "low", "close"]]

{'entrada_idx': 200000, 'direccion': 1, 'entrada': np.float64(1656.47), 'salida': np.float64(1656.375), 'motivo': 'SL', 'velas': 5, 'pnl': np.float64(-0.3450000000000273), 'R': np.float64(-0.5111111111111614)}


,open,high,low,close
datetime,,,,
2022-10-26 07:25:00,1656.47,1657.00,1655.98,1657.00
2022-10-26 07:30:00,1656.94,1657.31,1656.72,1657.05
2022-10-26 07:35:00,1657.08,1657.08,1656.50,1656.78
2022-10-26 07:40:00,1656.78,1657.11,1656.44,1656.50
2022-10-26 07:45:00,1656.47,1656.62,1656.10,1656.26
2022-10-26 07:50:00,1656.22,1656.37,1655.88,1656.25
2022-10-26 07:55:00,1656.25,1656.25,1655.44,1655.82
2022-10-26 08:00:00,1655.81,1656.57,1655.70,1656.12
2022-10-26 08:05:00,1656.19,1656.36,1655.60,1656.02


In [50]:
# 2) Control aleatorio con trailing: ¿la R media es ≈ −coste/SL, como debería?
rng = np.random.default_rng(0)
idxs = rng.choice(np.arange(300, len(df) - 30), 5000, replace=False)
dirs = rng.choice([1, -1], 5000)
R = [simular_trade_trailing(o, h, l, c, i, d, a[i-1], a[i-1], 24, 0.25)["R"] for i, d in zip(idxs, dirs)]
R = np.array(R)
print(f"R media: {R.mean():.3f}  |  mediana: {np.median(R):.3f}  |  % ganadores: {(R > 0).mean():.2f}")

R media: -0.202  |  mediana: -0.685  |  % ganadores: 0.30


In [51]:
from src.features import crear_dataset_libre
import time

t0 = time.time()
ds_libre = crear_dataset_libre(df, feats)
print(f"{len(ds_libre)} filas en {time.time() - t0:.0f} s")
ds_libre.to_parquet(ROOT / "data" / "processed" / "ds_libre.parquet")     # guardar: no recalcular

ds_libre["año"] = ds_libre["datetime"].dt.year
ds_libre.groupby("año")[["R_buy", "R_sell"]].mean().round(3)

159017 filas en 2 s


,R_buy,R_sell
año,,
2020,-0.218,-0.256
2021,-0.273,-0.272
2022,-0.236,-0.237
2023,-0.248,-0.263
2024,-0.148,-0.225
2025,-0.041,-0.107
2026,-0.033,-0.030


In [52]:
FEAT_L = ["z", "z_dia", "pend_dia", "atr_ratio", "ret_1", "ret_3", "ret_12",
          "rango_12", "coste_rel", "hora_sin", "hora_cos", "dia_sem"]

filas, elegidos_todos = [], []
for año in range(2022, 2027):
    inicio = pd.Timestamp(f"{año}-01-01")
    train = ds_libre[ds_libre["datetime"] < inicio - pd.Timedelta(days=2)]
    test = ds_libre[ds_libre["año"] == año].copy()
    ultimo = train[train["año"] == año - 1]

    m_buy = nuevo_regresor().fit(train[FEAT_L], train["R_buy"])
    m_sell = nuevo_regresor().fit(train[FEAT_L], train["R_sell"])

    # Umbral: top 1 % de la mejor predicción del último año de entrenamiento
    mejor_ult = np.maximum(m_buy.predict(ultimo[FEAT_L]), m_sell.predict(ultimo[FEAT_L]))
    umbral = np.quantile(mejor_ult, 0.99)

    pb, ps = m_buy.predict(test[FEAT_L]), m_sell.predict(test[FEAT_L])
    test["dir"] = np.where(pb >= ps, 1, -1)
    test["pred"] = np.maximum(pb, ps)
    test["R"] = np.where(test["dir"] == 1, test["R_buy"], test["R_sell"])
    el = test[test["pred"] > umbral]
    elegidos_todos.append(el)

    rng = np.random.default_rng(0)
    # (a) azar total: velas y direcciones al azar
    azar = []
    for _ in range(200):
        s = test.sample(len(el), random_state=int(rng.integers(1e9)))
        d = rng.choice([1, -1], len(s))
        azar.append(np.where(d == 1, s["R_buy"], s["R_sell"]).mean())
    # (b) placebo: mismas velas, dirección al azar
    plac = []
    for _ in range(200):
        d = rng.choice([1, -1], len(el))
        plac.append(np.where(d == 1, el["R_buy"], el["R_sell"]).mean())

    filas.append({"año": año, "elegidas": len(el), "R_ML": el["R"].mean(),
                  "R_azar": np.mean(azar), "p_azar": (np.array(azar) >= el["R"].mean()).mean(),
                  "R_placebo": np.mean(plac), "p_placebo": (np.array(plac) >= el["R"].mean()).mean()})

res_libre = pd.DataFrame(filas).set_index("año")
el_all = pd.concat(elegidos_todos)
r = el_all["R"]
print(f"Conjunto 2022–26: n={len(r)}  R={r.mean():.3f}  t={r.mean() / r.std() * np.sqrt(len(r)):.2f}  "
      f"mediana={r.median():.3f}  % ganadores={(r > 0).mean():.2f}")
res_libre.round(3)

Conjunto 2022–26: n=1682  R=0.014  t=0.23  mediana=-0.605  % ganadores=0.33


,elegidas,R_ML,R_azar,p_azar,R_placebo,p_placebo
año,,,,,,
2022,179,-0.359,-0.245,0.865,-0.230,0.935
2023,192,-0.135,-0.264,0.110,-0.252,0.190
2024,386,0.079,-0.180,0.000,-0.039,0.085
2025,377,0.016,-0.079,0.130,-0.052,0.225
2026,548,0.142,-0.032,0.000,0.089,0.280


In [53]:
diag = el_all.assign(año=el_all["datetime"].dt.year).groupby("año").agg(
    pct_compras=("dir", lambda d: (d == 1).mean()),
    R_ML=("R", "mean"),
    R_siempre_compra=("R_buy", "mean"),      # mismas velas, siempre comprando
    R_siempre_vende=("R_sell", "mean"))
diag.round(3)

,pct_compras,R_ML,R_siempre_compra,R_siempre_vende
año,,,,
2022,0.458,-0.359,-0.084,-0.368
2023,0.833,-0.135,-0.084,-0.420
2024,0.780,0.079,-0.029,-0.066
2025,0.875,0.016,-0.018,-0.065
2026,0.668,0.142,0.184,-0.005
